# PXD024871 — training the preregistered CNN

Runs the frozen specification (`SECTIONS.md` §§11–13) on the frozen dataset and
split. **Nothing here is a free parameter**: topology, encoding, grid, optimiser
and selection rule were all fixed before any model was fit, and the held-out
partition is guarded in code.

## Runtime: T4 GPU

This is the one stage that is genuinely matrix arithmetic. Runtime → Change
runtime type → **T4 GPU**. Measured on 4 CPU threads the full grid takes about
4.4 hours with early stopping; a T4 should be substantially faster, and cell 2
reports the measured rate so you can see what you actually got.

## Order, and why cell 3 cannot be skipped

1. Clone, record this machine as its own `env_id`
2. **Selftest** — encoding is lossless, gradients match finite differences, the
   network can fit a planted signal
3. CV grid, 16 configurations × 5 folds, resumable
4. Push results

The selftest is not a formality. It already caught an encoder that silently
deleted residues from 72.9% of the dataset, and a guard that would have let a
timing run unlock the held-out partition. If it fails, nothing downstream is
trustworthy — stop and report rather than proceeding.

## What this notebook will not do

It does not touch the test partition. `--mode test` refuses to run until a
complete grid has written a selection record, so the endpoint cannot be computed
before selection is finished. Reading it is a separate, deliberate act.

In [ ]:
#@title 1 — Clone, install, record this machine
from google.colab import userdata
import os, pathlib, subprocess

REPO   = "hafijbd/PXD024871_mapping"
BRANCH = "claude/mapping-research-project-prompt-glsess"
WORK   = pathlib.Path("/content/PXD024871_mapping")
ENV_ID = "env-003"  #@param {type:"string"}

os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
HELPER = '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'
GIT = ["git", "-c", f"credential.helper={HELPER}"]

if WORK.exists():
    subprocess.run(GIT + ["-C", str(WORK), "pull", "--rebase", "origin", BRANCH], check=True)
else:
    subprocess.run(GIT + ["clone", "--branch", BRANCH,
                          f"https://github.com/{REPO}.git", str(WORK)], check=True)
os.chdir(WORK)
subprocess.run(["git","config","user.name","Claude Code (Colab trainer)"], check=True)
subprocess.run(["git","config","user.email","noreply@anthropic.com"], check=True)

out = subprocess.run(["python3","-I","scripts/capture_environment.py",ENV_ID],
                     capture_output=True, text=True).stdout
if f"## {ENV_ID} " not in pathlib.Path("ENVIRONMENT.md").read_text():
    open("ENVIRONMENT.md","a").write(out)
print(out)

In [ ]:
#@title 2 — Confirm the accelerator
import torch, subprocess
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("=" * 64)
    print("  NO GPU. This stage is the one that benefits from one.")
    print("  Runtime > Change runtime type > T4 GPU, then re-run from cell 1.")
    print("  You can proceed on CPU; expect roughly 4-5 hours.")
    print("=" * 64)

In [ ]:
#@title 3 — Selftest. Do not skip; do not proceed if it fails.
r = subprocess.run(["python3","-I","scripts/train_cnn.py","--mode","selftest"],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit("SELFTEST FAILED — nothing downstream is trustworthy. Stop here.")

In [ ]:
#@title 4 — CV grid: 16 configurations x 5 folds. Resumable.
# If the session drops, re-run cells 1 -> 4; completed configurations are
# skipped. The held-out partition is never read in this mode.
!python3 -I scripts/train_cnn.py --mode cv

In [ ]:
#@title 5 — Push results
import glob, json, subprocess
subprocess.run(["git","add","results/model","ENVIRONMENT.md"], check=False)
st = subprocess.run(["git","status","--short"], capture_output=True, text=True).stdout
print(st or "(nothing to commit)")
if st.strip():
    n = len(json.load(open("results/model/cv_results.json"))) if \
        pathlib.Path("results/model/cv_results.json").exists() else 0
    subprocess.run(["git","commit","-m",
        f"CV grid on {ENV_ID}: {n} of 16 configurations complete\n\n"
        "Produced by scripts/train_cnn.py --mode cv. The held-out partition\n"
        "was not read. Selection record is written only by a complete grid.\n\n"
        "Co-Authored-By: Claude Opus 5.5 <noreply@anthropic.com>"], check=True)
    subprocess.run(GIT + ["push","origin",BRANCH], check=True)
    print("pushed")

## After the grid completes

`results/model/selection.json` names the selected configuration. It is written
**only** when all 16 configurations have run at the full epoch budget — a
truncated run leaves the held-out partition locked.

Computing the endpoint is a separate deliberate step, and it happens **once**:

```
python3 -I scripts/train_cnn.py --mode final   # selected config, 5 folds x 5 seeds
python3 -I scripts/train_cnn.py --mode test    # reads the held-out partition
```

Before running `--mode test`, be sure the selection is the one you intend to
report. The partition can be read honestly only once; a second reading after
seeing the first is no longer held out.

## What the result will mean

Fixed in advance, in `DECISION_LOG.md` D008 and the report's Discussion:

- The floor is **0.597**, not chance — a composition-only model reaches that
- Reject the null only if the **nominal-99%** lower bound exceeds **0.647**
- The primary test partition contains **15.99%** of peptides seen in training;
  the leakage-free subset is reported alongside and the gap is the inflation
- Power is **0.42** at a true AUROC of 0.70, so **failure to reject is not
  evidence of absent signal**
- Cross-platform transfer is preregistered: if performance collapses across
  instruments, the signal is substantially instrument rather than presentation